# Understat stats + merge with Transfermarkt — top 5 leagues, 2017–2025

Understat serves each (league, season) through one AJAX request
(`POST /main/getPlayersStats/`): no HTML scraping.

1. **Fetch** — 5 leagues × 9 seasons = 45 requests, each cached in
   `data/_understat/<league>_<year>.json`; a re-run only asks for what is missing.
2. **Link players** — `src/data/crosswalk.py`, deterministic rules, no fuzzy
   matching. Players are linked, not rows, and each link records its rule.
   Leftovers go to `data/manual/review_understat.csv` for a person to check;
   accepted ones go in `data/manual/id_overrides.csv` and win on the next run.
3. **Season-end club** — a player who moved mid-season is listed by both clubs on
   Transfermarkt. We keep the club they were at when the season ended, read from
   the club of their market-value snapshots.
4. **Join** — one row per player-season. Understat already counts a move inside a
   league as one row (team `"A,B"`); a move between two top-5 leagues is two
   Understat rows, summed here, so the stats are always the whole season's.

Output: `data/understat_stats.parquet` (raw stats, all leagues) and
`data/panel_final.parquet` (Transfermarkt + Understat, one row per player-season).

In [ ]:
import json
import os
import sys
import time
from pathlib import Path

import pandas as pd
import requests


def find_project_root(marker: str = "src/features/engineer.py") -> Path:
    """Walk up from the working directory to the repository root."""
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / marker).exists():
            return candidate
    raise RuntimeError(f"could not find {marker} in {here} or above")


BASE_PATH = Path(os.environ.get("TRANSFER_EDGE_ROOT") or find_project_root())
sys.path.insert(0, str(BASE_PATH))

from src.data.crosswalk import MANUAL_DIR, build_crosswalk  # noqa: E402
from src.data.leagues import LEAGUES  # noqa: E402

DATA_DIR = BASE_PATH / "data"
CACHE_DIR = DATA_DIR / "_understat"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

TM_PATH = DATA_DIR / "players_panel.parquet"
HISTORY_PATH = DATA_DIR / "_value_histories.csv"
UNDERSTAT_PATH = DATA_DIR / "understat_stats.parquet"
PANEL_PATH = DATA_DIR / "panel_final.parquet"

YEARS = list(range(2017, 2026))   # same seasons as notebook 01
REQUEST_DELAY = 1.5               # seconds between requests

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 "
        "(KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    ),
    "Referer": "https://understat.com/",
    "X-Requested-With": "XMLHttpRequest",
    "Content-Type": "application/x-www-form-urlencoded",
}

print(f"project root: {BASE_PATH}")
print("leagues:", ", ".join(f"{c} ({lg.understat})" for c, lg in LEAGUES.items()))

## 1. Fetch

Understat uses the season's start year (2017 = 2017-18), like Transfermarkt's
`saison_id`. A season Understat does not have yet is reported, and its
Transfermarkt rows simply get no stats.

In [ ]:
def get_understat_season(code: str, year: int) -> list[dict]:
    """All players of one league-season, from the cache or one request."""
    path = CACHE_DIR / f"{code}_{year}.json"
    if path.exists():
        return json.loads(path.read_text())
    resp = requests.post(
        "https://understat.com/main/getPlayersStats/",
        headers=HEADERS,
        data={"league": LEAGUES[code].understat, "season": str(year)},
        timeout=30,
    )
    resp.raise_for_status()
    players = resp.json().get("players") or []
    if players:   # an empty answer is not cached: the season may appear later
        path.write_text(json.dumps(players))
    time.sleep(REQUEST_DELAY)
    return players


frames, no_data = [], []
for code in LEAGUES:
    for year in YEARS:
        players = get_understat_season(code, year)
        if not players:
            no_data.append((code, year))
            continue
        frames.append(pd.DataFrame(players).assign(year=year, league_code=code))

understat_raw = pd.concat(frames, ignore_index=True)
print(understat_raw.groupby(["league_code", "year"]).size().unstack())
if no_data:
    print(f"no Understat data for: {no_data} -> those Transfermarkt rows get no stats")

## 2. Clean

In [ ]:
NUMERIC_COLS = ["games", "minutes", "goals", "assists", "shots", "key_passes",
                "yellow_cards", "red_cards", "npg", "xG", "xA", "npxG",
                "xGChain", "xGBuildup"]

understat_df = understat_raw.rename(columns={
    "id": "understat_id",
    "player_name": "player_name_us",
    "team_title": "team_us",
    "position": "position_us",
    "time": "minutes",
})
understat_df["understat_id"] = understat_df["understat_id"].astype(str)
for col in NUMERIC_COLS:
    understat_df[col] = pd.to_numeric(understat_df[col], errors="coerce")
understat_df["league"] = understat_df["league_code"].map(lambda c: LEAGUES[c].name)

understat_df.to_parquet(UNDERSTAT_PATH, index=False)
print(f"saved {UNDERSTAT_PATH.name}: {understat_df.shape}")
print(f"ids in two leagues in one season (moved between leagues): "
      f"{understat_df.duplicated(['year', 'understat_id']).sum()}")

## 3. Link players (deterministic crosswalk)

All rules compare players inside the same league-season and are one-to-one; see
the docstring of `src/data/crosswalk.py`. In short: same name + same club, then
one name inside the other in the same club ("Kepa" / "Kepa Arrizabalaga"), then
surname + first initial in the same club, and last a name unique in the whole
league-season. Clubs are paired by the players their squads share, not by text.

The full Transfermarkt panel goes in — both clubs of a mid-season move — so
either club can confirm the link.

In [ ]:
tm_panel = pd.read_parquet(TM_PATH)
tm_panel["player_id"] = tm_panel["player_id"].astype(str)
tm_panel["year"] = tm_panel["year"].astype(int)

xw = build_crosswalk(tm_panel, understat_df, fbref=None)
links = xw["crosswalk"]
review = xw["review_understat"]

played = understat_df[understat_df["minutes"] > 0]
linked = played["understat_id"].isin(set(links["src_id"]))
print(f"links: {len(links):,} players   by rule: {links['rule'].value_counts().to_dict()}")
print(f"clubs not paired : {xw['unmapped_clubs_understat'] or 'none'}")
print(f"conflicts dropped: {len(xw['conflicts_understat'])}")
print("\nUnderstat player-seasons with minutes that got a link, by league:")
print(linked.groupby(played["league_code"]).mean().map("{:.1%}".format).to_string())
print(f"\nleft for review: {len(review)} players -> {MANUAL_DIR / 'review_understat.csv'}")
print(review.groupby("league_code").size().to_string())

## 4. Season-end club

Transfermarkt lists a mid-season mover in both squads. Understat's numbers cover
the whole season, so copying them to both rows would count every minute twice.
We keep one row: the club they ended the season at.

The value API stores the player's club with every snapshot (`club_id`). The
club names in the panel are mapped to those ids by majority vote over players
with one club that season, then for each mover we walk their snapshots of that
season (August–June) from the newest back, and keep the first club that is one
of their rows. Without a usable snapshot we fall back to the scrape order.

In [ ]:
histories = pd.read_csv(HISTORY_PATH, parse_dates=["date"],
                        dtype={"player_id": str, "club_id": "Int64"})


def club_ids(panel: pd.DataFrame, histories: pd.DataFrame) -> pd.Series:
    """(league_code, club) -> Transfermarkt club id, by vote over one-club player-seasons."""
    single = panel[~panel.duplicated(["player_id", "year"], keep=False) & panel["value_date"].notna()]
    m = single.merge(histories[["player_id", "date", "club_id"]].dropna(),
                     left_on=["player_id", "value_date"], right_on=["player_id", "date"])
    votes = (m.groupby(["league_code", "club", "club_id"]).size().rename("n").reset_index()
             .sort_values(["n", "club_id"], ascending=[False, True]))
    return votes.drop_duplicates(["league_code", "club"]).set_index(["league_code", "club"])["club_id"]


def season_end_rows(panel: pd.DataFrame, histories: pd.DataFrame) -> tuple[pd.DataFrame, dict]:
    """One row per player-season: the club they were at when the season ended."""
    ids = club_ids(panel, histories)
    panel = panel.assign(_club_id=[ids.get(k) for k in zip(panel["league_code"], panel["club"])],
                         _order=range(len(panel)))
    multi = panel.duplicated(["player_id", "year"], keep=False)
    snaps = histories.dropna(subset=["club_id"]).sort_values("date")
    snaps = snaps[snaps["player_id"].isin(set(panel.loc[multi, "player_id"]))]
    by_player = dict(tuple(snaps.groupby("player_id")))

    keep, how = [], {"last snapshot": 0, "earlier snapshot": 0, "scrape order": 0}
    for (pid, year), g in panel[multi].groupby(["player_id", "year"]):
        s = by_player.get(pid, snaps.iloc[:0])
        s = s[(s["date"] >= f"{year}-08-01") & (s["date"] <= f"{year + 1}-06-30")]
        pick, rule = g["_order"].idxmax(), "scrape order"
        for i, club in enumerate(reversed(s["club_id"].tolist())):   # newest first
            hit = g[g["_club_id"] == club]
            if len(hit) == 1:
                pick, rule = hit.index[0], "last snapshot" if i == 0 else "earlier snapshot"
                break
        keep.append(pick)
        how[rule] += 1
    out = pd.concat([panel[~multi], panel.loc[keep]]).sort_values("_order")
    return out.drop(columns=["_club_id", "_order"]).reset_index(drop=True), how


ids = club_ids(tm_panel, histories)
missing = sorted(set(zip(tm_panel["league_code"], tm_panel["club"])) - set(ids.index))
print(f"clubs with a Transfermarkt id: {len(ids)}   without: {missing or 'none'}")

multi = tm_panel.duplicated(["player_id", "year"], keep=False)
moves = tm_panel[multi].groupby(["player_id", "year"])["league_code"].nunique()
print(f"player-seasons with two clubs: {len(moves):,}   ({(moves > 1).sum():,} across two leagues)")

tm_df, how = season_end_rows(tm_panel, histories)
print(f"season-end club chosen by: {how}")
print(f"{len(tm_panel):,} rows -> {len(tm_df):,} (one per player-season)")

## 5. Join

Understat stats are grouped by Transfermarkt player and season. Counts (minutes,
goals, xG, …) are summed over the leagues they played in that season; name and
position come from their row with the most minutes. `us_leagues` lists the
leagues the stats come from, and `us_rule` the crosswalk rule behind the link.

A Transfermarkt row without stats is a player who did not play in the top 5 that
season (reserves, players loaned outside the five leagues) or one still in the
review file.

In [ ]:
def understat_by_player(us: pd.DataFrame, links: pd.DataFrame) -> pd.DataFrame:
    """One row per (Transfermarkt player, season), summed over leagues."""
    us = us.merge(links[["src_id", "player_id", "rule"]], left_on="understat_id", right_on="src_id")
    us = us.sort_values(["minutes", "league_code"], ascending=[False, True])
    g = us.groupby(["player_id", "year"], sort=False)
    out = g[["understat_id", "player_name_us", "position_us", "rule"]].first()
    out = out.join(g[NUMERIC_COLS].sum(min_count=1))
    out["team_us"] = g["team_us"].agg(",".join)
    out["us_leagues"] = g["league_code"].agg(lambda s: ",".join(sorted(set(s))))
    return out.rename(columns={"rule": "us_rule"}).reset_index()


stats = understat_by_player(understat_df, links)
panel_final = tm_df.merge(stats, on=["player_id", "year"], how="left")

has = panel_final["understat_id"].notna()
print(f"panel_final: {panel_final.shape}")
print("rows with Understat stats, by league:")
print(has.groupby(panel_final["league"]).mean().map("{:.1%}".format).to_string())
print(f"\nstats summed over two leagues: {panel_final['us_leagues'].str.contains(',', na=False).sum()} rows")

## 6. Checks

In [ ]:
dup_rows = panel_final.duplicated(["year", "player_id"]).sum()
reused = (panel_final[has].groupby(["year", "understat_id"]).size() > 1).sum()
too_many = (panel_final["minutes"] > 38 * 90 + 600).sum()   # 38 full league games + slack

# Every linked Understat player-season with minutes should land on a panel row,
# unless Transfermarkt has no row for that player in that season.
linked_us = played.merge(links[["src_id", "player_id"]], left_on="understat_id", right_on="src_id")
in_tm = linked_us.set_index(["player_id", "year"]).index.isin(tm_df.set_index(["player_id", "year"]).index)
landed = linked_us[in_tm].set_index(["player_id", "year"]).index.isin(
    panel_final[has].set_index(["player_id", "year"]).index)

print(f"duplicate player-seasons      : {dup_rows}")
print(f"Understat id on two players   : {reused}")
print(f"seasons over {38 * 90 + 600} minutes       : {too_many}")
print(f"linked stats with no TM season: {(~in_tm).sum()}  (TM roster lacks the player that year)")
print(f"linked stats that landed      : {landed.mean():.1%}")

assert dup_rows == 0, "one row per player-season expected"
assert reused == 0, "an Understat id is attached to two Transfermarkt players"
assert landed.all(), "a linked player-season lost its stats in the join"
print("\nOK")

In [ ]:
panel_final.to_parquet(PANEL_PATH, index=False)
print(f"saved {PANEL_PATH.name}: {panel_final.shape}")
print(f"columns: {list(panel_final.columns)}")
(panel_final.groupby("league")
 .agg(rows=("player_id", "size"), players=("player_id", "nunique"),
      with_stats=("understat_id", lambda s: f"{s.notna().mean():.0%}"),
      median_minutes=("minutes", "median"))
 .sort_index())